# P02 EHR preprocessing and basic modelling

CDT in Healthcare Data Science, ML for Time Series. Tuesday 3 November, 13:00 to 14:15.

In this practical, we will predict in-hospital mortality from the first 48 hours of ICU data. Some measurements in the data are missing, so we start by finding them and filling them in (imputation). We then train two classical models on summary features, and the three sequence models from this morning's lecture: an LSTM, a TCN and a Transformer.

## Your tasks
1. Part 1 (25 min): find the missing values, write two imputers and compare them.
2. Part 2 (10 min): summary features, logistic regression and a random forest.
3. Part 3 (30 min): an LSTM, a TCN and a Transformer encoder.
4. Part 4 (10 min): compare all the models.

The cells marked **TODO** are for you to complete, and each one is followed by a few checks that print PASS or FAIL. Later cells use your code, so please do the TODOs in order. If you finish early, there are some extensions at the end.

### Before you start
1. Run the three cells under *Set-up* below, in order. Ideally do this before the session; it takes about 5 minutes.
   - The environment check prints the version of each package, and installs anything that is missing (in Colab, usually only `pyzipper`).
   - The imports cell loads the packages and defines a few helper functions, such as `check()`.
   - The data cell downloads `EHR_data.zip` (about 10 MB) from Google Drive and asks for the data password. You can find the password on Canvas, on the [HDS-M04: Module Info](https://canvas.ox.ac.uk/courses/298593/pages/hds-m04-module-info-machine-learning-for-time-series-2026?module_item_id=3212983) page, in the section "Practical 2 – EHR Preprocessing and Basic Modelling" (or simply search the page for "passcode"). Type it into the box under the cell and press Enter. If it does not work, ask a tutor for the password.

   If all three run without an error, you are ready. This is the recommended way to do this lab: in Colab, with the data downloaded by the data cell. The P.S. notes at the end of this section are only for reference, in case the download fails or you want to use your own Python environment (which we do not recommend for this session).
2. You do not need a GPU: the whole notebook runs on a CPU in a few minutes.
3. Make sure you save your own copy of this notebook in your Google Drive (*File > Save a copy in Drive*), and work in that copy. If you do not, your changes will be lost when the Colab session ends.

P.S. If the download fails, download `EHR_data.zip` in your browser, from [Google Drive](https://drive.google.com/file/d/1obrZjLQAyx0iJXuGusKGUUTpaIfrXGd_/view?usp=sharing) or from [Canvas](https://canvas.ox.ac.uk/courses/298593/files/9308218?wrap=1), and then either
- drag it into the *Files* panel on the left (the folder icon) and re-run the data cell. The uploaded file is lost when the Colab session ends; or
- put it, still zipped, in a folder called `cdt_ehr_lab` at the top level of your Google Drive, set `USE_MY_DRIVE = True` in the data cell, and re-run it. Colab will ask for access to your Drive.

P.S. If you are not using Colab, you can set up your own environment with conda, and start Jupyter in the folder with this notebook:
```bash
conda create -n ehr-lab python
conda activate ehr-lab
pip install numpy pandas matplotlib scikit-learn imbalanced-learn torch gdown pyzipper jupyter
jupyter notebook
```
The data cell unzips the data into an `EHR_data/` folder next to the notebook. If the download fails, put `EHR_data.zip` next to the notebook yourself and keep it zipped, as your computer's own unzip tool may not be able to open it.

The dataset is provided for teaching only. Please keep it in your own Drive and Colab session, and do not upload it anywhere else.

Lab designed by: Yujiang Wang (yujiang.wang@eng.ox.ac.uk)

## Set-up
Run the three cells below in order.

In [ ]:
### check the environment: print the version of each package, and install the missing ones
import importlib
import subprocess
import sys

# import name: pip name
packages = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn',
            'imblearn': 'imbalanced-learn', 'torch': 'torch', 'gdown': 'gdown', 'pyzipper': 'pyzipper'}
for module, package in packages.items():
    try:
        version = importlib.import_module(module).__version__
    except ImportError:
        print(f'installing {package} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
        importlib.invalidate_caches()
        version = importlib.import_module(module).__version__
    print(f'{package:17s} {version}')

In [ ]:
### load packages
import copy
import math
import os
import shutil
import time
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn.metrics as metrics
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.nn import functional as F

warnings.filterwarnings('ignore')
SEED = 0   # random seed, so that we all get the same results
np.random.seed(SEED)
torch.manual_seed(SEED)

# use a GPU if there is one
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'Using device: {device}')


# prints PASS or FAIL for the checks after each TODO
def check(description, passed, hint=''):
    print(('PASS  ' if passed else 'FAIL  ') + description)
    if not passed and hint:
        print('      hint:', hint)


# compute a few scores from the predicted probabilities of class 1 (mortality)
def compute_scores(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {'confusion': metrics.confusion_matrix(y_true, y_pred),
            'accuracy': metrics.accuracy_score(y_true, y_pred),
            'balanced_accuracy': metrics.balanced_accuracy_score(y_true, y_pred),
            'F1': metrics.f1_score(y_true, y_pred),
            'AUROC': metrics.roc_auc_score(y_true, y_prob),
            'AUPRC': metrics.average_precision_score(y_true, y_prob)}


def print_scores(s):
    print(f"Accuracy: {s['accuracy']:.3f} | Balanced accuracy: {s['balanced_accuracy']:.3f} | F1: {s['F1']:.3f} | "
          f"AUROC: {s['AUROC']:.3f} | AUPRC: {s['AUPRC']:.3f}")

In [ ]:
### load the data
DATA_URL = 'https://drive.google.com/file/d/1obrZjLQAyx0iJXuGusKGUUTpaIfrXGd_/view?usp=sharing'
ZIP_NAME = 'EHR_data.zip'
USE_MY_DRIVE = False   # set to True to read MyDrive/cdt_ehr_lab/EHR_data.zip instead of downloading it
# the data password (see "Before you start"): leave it as None and the cell will ask you for it,
# or replace None with the password, in quotes
DATA_PASSWORD = None

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

work_dir = Path('/content') if IN_COLAB else Path(os.environ.get('EHR_LAB_DIR', '.'))
data_dir = work_dir / 'EHR_data'   # the data files end up here, e.g. EHR_data/X_data.npy

if not (data_dir / 'X_data.npy').is_file():
    # look for the zip: uploaded to the Colab Files panel (or next to the notebook), or in your own Drive
    zip_path = work_dir / ZIP_NAME
    if not zip_path.is_file() and IN_COLAB and USE_MY_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        zip_path = Path('/content/drive/MyDrive/cdt_ehr_lab') / ZIP_NAME

    # otherwise, download it from Google Drive
    if not zip_path.is_file():
        import re
        import gdown
        file_id = re.search(r'(?:/d/|id=)([\w-]+)', DATA_URL).group(1)
        try:
            gdown.download(id=file_id, output=str(zip_path), quiet=False)
        except Exception as e:
            print('Download failed:', e)
        if zip_path.is_file() and not zipfile.is_zipfile(zip_path):
            zip_path.unlink()   # Google sometimes sends back an error page instead of the zip
    if not zip_path.is_file():
        raise FileNotFoundError(f'Could not get {ZIP_NAME}; see "If the download fails" at the top, then re-run this cell.')

    # unzip it into a temporary folder; the zip is password-protected
    import pyzipper   # like zipfile, but it can also open AES-encrypted zips
    tmp_dir = work_dir / 'EHR_data_unzipping'
    shutil.rmtree(tmp_dir, ignore_errors=True)
    with pyzipper.AESZipFile(zip_path) as zf:
        if any(info.flag_bits & 0x1 for info in zf.infolist()):
            if not DATA_PASSWORD:
                from getpass import getpass
                DATA_PASSWORD = getpass('Data password (on Canvas, or ask a tutor): ').strip()
            zf.setpassword(DATA_PASSWORD.encode())
        try:
            zf.extractall(tmp_dir)
        except (RuntimeError, zipfile.BadZipFile):
            shutil.rmtree(tmp_dir, ignore_errors=True)   # clean up, so that a re-run starts again
            raise RuntimeError('Could not unzip the data: wrong password? Re-run this cell to try again, '
                               'or ask a tutor for the password.') from None

    # move the folder that holds the data files to EHR_data/, wherever it is in the zip, and delete the rest
    found = sorted(tmp_dir.rglob('X_data.npy'))
    assert found, 'X_data.npy is not in the zip; please ask a tutor'
    shutil.rmtree(data_dir, ignore_errors=True)
    shutil.move(str(found[0].parent), str(data_dir))
    shutil.rmtree(tmp_dir, ignore_errors=True)

X = np.load(data_dir / 'X_data.npy').astype(np.float32)
y = np.load(data_dir / 'y_data.npy').reshape(-1).astype(np.int64)
feature_names = np.array((data_dir / 'feature_names.txt').read_text(encoding='utf-8').splitlines()[1:])

print(f'X shape (N x T x P): {X.shape}')
print(f'y shape: {y.shape}, mortality rate: {100 * y.mean():.1f}%')
print(f'number of features: {len(feature_names)}')

The data (`X_data.npy`) is stored as $\mathbf{X} \in \mathbb{R}^{N\times T \times P}$: $N = 9{,}622$ ICU patients, $T = 48$ hourly time steps and $P = 59$ features. The labels (`y_data.npy`) are in-hospital mortality, $\mathbf{y} \in \{0,1\}^{N}$; 17% of the patients died.

The features are vital signs and lab values (heart rate, blood pressure, respiratory rate, temperature, pH and so on), some demographics (sex, height, weight), the ICU type, and the Glasgow Coma Scale (GCS), whose eye, verbal and motor sub-scores are one-hot coded. The continuous features are already standardised.

We have kept some missing values (`NaN`) in this dataset on purpose, for this practical: about 5% of the patients have some of their measurements missing, just like the gaps you will find in real EHRs.

## Part 1: missing values and imputation (25 min)

### TODO 1: where are the missing values?
Find
- which patients have at least one missing value, as a boolean vector of length $N$ (`patient_has_nan`);
- which features have missing values, as an array of their names (`features_with_nan`).

<b>Hint</b>: `np.isnan(X)` gives a boolean array of the same shape as `X`; combine it with `.any(axis=...)`. You can pass a tuple such as `axis=(1, 2)` to reduce over several axes at once.

In [ ]:
nan_mask = np.isnan(X)
print(f'missing values: {nan_mask.sum()} ({100 * nan_mask.mean():.3f}% of all entries)')

############################################################
# TODO 1: write your code here.
# patient_has_nan: a boolean array of shape (N,), True for the patients with at least one NaN
# features_with_nan: an array with the names of the features that have at least one NaN
patient_has_nan = None
features_with_nan = None

raise NotImplementedError('TODO 1: write your code above, then delete this line')
############################################################

In [ ]:
print(f'patients with >= 1 NaN: {patient_has_nan.sum()} of {len(y)} ({100 * patient_has_nan.mean():.1f}%)')
print('features with NaNs:', [str(f) for f in features_with_nan])

check('patient_has_nan has one entry per patient', np.shape(patient_has_nan) == y.shape,
      'reduce over the time and feature axes: .any(axis=(1, 2))')
check('about 5% of the patients have missing values', 0.04 < np.mean(patient_has_nan) < 0.06)
check('features_with_nan lists the 10 features with missing values', len(features_with_nan) == 10,
      'reduce over the patient and time axes: .any(axis=(0, 1)), then index feature_names')

Let's look at one patient with missing values. Each black cell is a missing measurement.

In [ ]:
### where are the missing values of one patient?
miss_cols = np.where(nan_mask.any(axis=(0, 1)))[0]
example = np.where(nan_mask.any(axis=(1, 2)))[0][0]   # the first patient with a NaN

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7))
ax1.imshow(nan_mask[example][:, miss_cols].T, aspect='auto', cmap='Greys', interpolation='nearest')
ax1.set_yticks(range(len(miss_cols))); ax1.set_yticklabels(feature_names[miss_cols])
ax1.set_xlabel('Time step (hour)'); ax1.set_title(f'Patient {example}: missing entries (black)')

# the heart rate of the same patient; the red lines mark the missing hours
hr = X[example, :, feature_names == 'Heart Rate'].squeeze()
ax2.plot(hr, 'o-', label='observed heart rate')
for t in np.where(np.isnan(hr))[0]:
    ax2.axvline(t, color='r', alpha=0.4)
ax2.set_xlabel('Time step (hour)'); ax2.set_ylabel('Heart rate (standardised)')
ax2.set_title('Heart rate: red lines mark missing hours'); ax2.legend()
fig.tight_layout(); plt.show()

# is the mortality rate different for the patients with missing values?
has_missing = nan_mask.any(axis=(1, 2))
print(f'mortality rate, patients with missing values: {100 * y[has_missing].mean():.1f}% | '
      f'other patients: {100 * y[~has_missing].mean():.1f}%')

**Discuss:** the mortality rate is about the same for patients with and without missing values, and the gaps look scattered at random. This is called **missing completely at random (MCAR)**: whether a value is missing tells us nothing about the patient.

Real EHR data are rarely like this. A lactate is measured *because* a clinician is worried, and stable patients are measured less often at night. When the missingness depends on the patient (missing at random, or missing not at random), what was measured, and when, is informative in itself.

### What do NaNs do to a standard pipeline?

In [ ]:
from sklearn.linear_model import LogisticRegression

X_m = np.mean(X, axis=1)   # (N, P): the mean of each feature over time
print('patients with a NaN in their mean features:', np.isnan(X_m).any(axis=1).sum())
try:
    LogisticRegression().fit(X_m, y)
except ValueError as e:
    print('LogisticRegression failed:', str(e).splitlines()[0])

One `NaN` in a series makes that patient's mean `NaN` as well. Most scikit-learn models refuse to train on it, and in PyTorch a single `NaN` in a batch turns the loss, the gradients and then all the weights into `NaN`. So what can we do?

1. Drop the patients with missing values. This is easy, but we lose data (5% here, and much more in real EHRs), and the patients we keep may no longer be representative.
2. Ignore the gaps when we summarise a series, e.g. with `np.nanmean`. This works for summary features, but a sequence model needs a value at every time step.
3. Impute, i.e. fill in every missing value. This is what we do next.
4. Tokenise: describe a patient by the events that were actually observed, so there is nothing to fill in.

### Split before you impute
Imputation uses statistics estimated from the data, such as a feature's mean, just like a model's weights. If we compute them on all the patients, information from the test set leaks into training. So we first split the patients, stratified by outcome, into 70% for training, 15% for validation and 15% for testing.

In [ ]:
### split the patients into train/validation/test (70%/15%/15%), stratified by outcome
idx_all = np.arange(len(y))
idx_train, idx_temp = train_test_split(idx_all, test_size=0.30, random_state=SEED, stratify=y)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=SEED, stratify=y[idx_temp])
print(f'patients: train {len(idx_train)} | validation {len(idx_val)} | test {len(idx_test)}')
print(f'mortality: train {100 * y[idx_train].mean():.1f}% | validation {100 * y[idx_val].mean():.1f}% | '
      f'test {100 * y[idx_test].mean():.1f}%')

# the mean of each feature over all training patients and time steps, ignoring NaNs
train_mean = np.nanmean(X[idx_train], axis=(0, 1))

### TODO 2: mean imputation
Replace each missing value of feature $p$ with the mean of that feature on the training set, $\bar{x}_p$:
$$
\hat{x}_{n,t,p}=\begin{cases}
            \bar{x}_p, & \text{if $x_{n,t,p}$ is missing}\\
            x_{n,t,p}, & \text{otherwise}
         \end{cases}
$$

<b>Hint</b>: `np.where(np.isnan(X))` returns three arrays of indices (patient, time, feature), with one entry for each NaN.

In [ ]:
# replace every NaN in feature p with train_mean[p]
def impute_mean(X, train_mean):
    X = X.copy()   # work on a copy, so that the original X keeps its NaNs
    ############################################################
    # TODO 2: write your code here.
    # Replace every NaN in feature p of X with train_mean[p], and return X.

    raise NotImplementedError('TODO 2: write your code above, then delete this line')
    ############################################################

In [ ]:
X_mean = impute_mean(X, train_mean)
check('no NaNs left', not np.isnan(X_mean).any())
check('observed values unchanged', np.allclose(X_mean[~nan_mask], X[~nan_mask]),
      'only write to the positions where X is NaN')
check('each gap is filled with its feature\'s training mean',
      np.allclose(X_mean[nan_mask], np.broadcast_to(train_mean, X.shape)[nan_mask]))
check('the input X is not modified in place', np.isnan(X).sum() == nan_mask.sum(), 'start with X = X.copy()')

### TODO 3: last observation carried forward (LOCF)
Mean imputation ignores time; LOCF uses it. For each patient and feature, a missing value is replaced by the last value observed before it. A gap at the start of a series has no earlier value, so we fill it backwards with the next observed value instead. If a whole series is missing, use the training mean (you can reuse `impute_mean`).

<b>Hint</b>: `pandas.DataFrame.ffill()` and `.bfill()` fill along the rows of a table. For one feature `p`, `pd.DataFrame(X[:, :, p].T)` is a `(T, N)` table with time along the rows.

In [ ]:
# forward-fill within each patient and feature, back-fill the gaps at the start,
# then use the training mean for anything that is still missing
def impute_locf(X, train_mean):
    X = X.copy()   # work on a copy, so that the original X keeps its NaNs
    ############################################################
    # TODO 3: write your code here.
    # Fill the NaNs in X forward along time, then backward, and then with the training mean
    # for the series that are completely missing. Return X, with no NaNs left.

    raise NotImplementedError('TODO 3: write your code above, then delete this line')
    ############################################################

In [ ]:
toy = np.array([np.nan, 1, np.nan, np.nan, 4, np.nan], dtype=np.float32).reshape(1, 6, 1)
toy_out = impute_locf(toy, np.array([0.0], dtype=np.float32)).ravel()
check(f'toy series [nan, 1, nan, nan, 4, nan] -> [1, 1, 1, 1, 4, 4] (yours: {toy_out.tolist()})',
      np.allclose(toy_out, [1, 1, 1, 1, 4, 4]), 'forward-fill, then back-fill the leading gap')
toy_out = impute_locf(np.full((1, 3, 1), np.nan, dtype=np.float32), np.array([7.0], dtype=np.float32)).ravel()
check('a fully missing series falls back to the training mean', np.allclose(toy_out, 7))

X_locf = impute_locf(X, train_mean)
check('no NaNs left on the real data', not np.isnan(X_locf).any())
check('observed values unchanged', np.allclose(X_locf[~nan_mask], X[~nan_mask]))

We also give you a third imputer: linear interpolation in time, between the values before and after each gap.

In [ ]:
### a third imputer: linear interpolation in time; the values at either end are held constant
def impute_linear(X, train_mean):
    X = X.copy()
    for p in range(X.shape[2]):
        series = pd.DataFrame(X[:, :, p].T)
        X[:, :, p] = series.interpolate(method='linear', limit_direction='both').to_numpy().T
    return impute_mean(X, train_mean)

X_lin = impute_linear(X, train_mean)

# compare the three imputers on the heart rate of the same patient
f = np.where(feature_names == 'Heart Rate')[0][0]
gaps = np.where(nan_mask[example, :, f])[0]
plt.figure(figsize=(12, 4))
plt.plot(X[example, :, f], 'ko-', label='observed', zorder=3)
for name, Xi, m in [('mean', X_mean, 's'), ('LOCF', X_locf, '^'), ('linear', X_lin, 'v')]:
    plt.plot(gaps, Xi[example, gaps, f], m, ms=10, label='imputed: ' + name)
plt.xlabel('Time step (hour)'); plt.ylabel('Heart rate (standardised)')
plt.title(f'Patient {example}: imputed values at the missing hours'); plt.legend()
plt.show()

### Which imputer is best? Hold-out masking
We cannot score an imputer on the values that are really missing, because we never saw them. Instead, we hide some of the values that we did observe, impute them, and compare the imputed values with the true ones.

The cell below hides 5% of the observed values of the test patients, runs each imputer, and reports the mean absolute error (MAE) for each of the 10 features with missing values. We use the MAE rather than the RMSE, as a few extreme outliers would dominate a squared error. Run it, then discuss:
- Which imputer is best overall? Is it the best one for every feature?
- Why is LOCF almost perfect for *Height* and *Weight*? Have a look at a few patients' series for these two features.
- When could mean imputation still be a sensible choice?

In [ ]:
# hide a random fraction of the observed values; returns the masked copy and the positions we hid
def holdout_mask(X, frac=0.05, seed=0):
    rng = np.random.default_rng(seed)
    hide = ~np.isnan(X) & (rng.random(X.shape) < frac)
    X_hidden = X.copy()
    X_hidden[hide] = np.nan
    return X_hidden, hide

cont = np.where(nan_mask.any(axis=(0, 1)))[0]   # the 10 features with missing values (all continuous)
X_true = X[idx_test]
X_hidden, hidden = holdout_mask(X_true, frac=0.05)

# MAE of each imputer on the hidden values, for each feature
mae = {}
for name, imputer in [('mean', impute_mean), ('LOCF', impute_locf), ('linear', impute_linear)]:
    X_imp = imputer(X_hidden, train_mean)
    mae[name] = [np.mean(np.abs(X_imp[:, :, p][hidden[:, :, p]] - X_true[:, :, p][hidden[:, :, p]])) for p in cont]
MAE = pd.DataFrame(mae, index=feature_names[cont])
MAE.loc['(average)'] = MAE.mean()
MAE.round(3)

## Part 2: summary features and off-the-shelf models (10 min)

A simple baseline, which is often hard to beat, summarises each patient's 48-hour series with a few statistics per feature: the mean $\bar{\mathbf{x}}$, the standard deviation $\sigma$, and the change from the first to the last hour divided by $\sigma$, $\Delta$. This gives a vector of $3P = 177$ features for each patient.

In [ ]:
### summary features: the mean, std and (scaled) change of each feature over the 48 hours
def summary_features(X):
    X_m, X_sd = X.mean(axis=1), X.std(axis=1)
    X_delta = (X[:, -1, :] - X[:, 0, :]) / X_sd
    X_delta[~np.isfinite(X_delta)] = 0   # a constant series has sd = 0
    return np.concatenate((X_m, X_sd, X_delta), axis=1)

F_names = np.concatenate((feature_names + ' (mean)', feature_names + ' (std)', feature_names + ' (delta)'))
F_all = summary_features(X_locf)   # we use the LOCF-imputed data from TODO 3
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]
print('summary features:', F_all.shape)

### TODO 4: normalisation without leakage
Standardise each feature with the z-score, $z = (x - \mu)/\sigma$. As with the imputation, $\mu$ and $\sigma$ must come from the **training set only**, and are then applied to both the training and the test set. Be careful with features that have $\sigma = 0$.

<b>Hint</b>: `F_train.mean(axis=0)` and `F_train.std(axis=0)` give one value for each feature (column). Before dividing, replace the standard deviations that are 0 with 1, so that a constant feature becomes 0 instead of NaN.

In [ ]:
# standardise both sets with the mean and std of the training set
def zscore(F_train, F_test):
    ############################################################
    # TODO 4: write your code here.
    # Return two arrays, F_train and F_test, both standardised with the mean and std of F_train.

    raise NotImplementedError('TODO 4: write your code above, then delete this line')
    ############################################################

In [ ]:
F_train, F_test = zscore(F_all[idx_train], F_all[idx_test])
check('training features have mean 0 and std 1 (or 0 for constant features)',
      np.allclose(F_train.mean(axis=0), 0, atol=1e-4) and np.all(np.isclose(F_train.std(axis=0), 1, atol=1e-3) | np.isclose(F_train.std(axis=0), 0)))
a, b = zscore(np.array([[0.0], [2.0]]), np.array([[100.0]]))
check('the test set uses the training mean and std (toy: train [0, 2], test [100] -> 99)', np.isclose(b[0, 0], 99),
      'compute mu and sd from F_train only, then apply them to F_test')
check('a constant feature does not produce NaN', np.isfinite(zscore(np.ones((3, 1)), np.ones((1, 1)))[0]).all())

EHRs also contain values that cannot be right, e.g. from unit errors, typos or sensor artefacts; a few entries here are hundreds of standard deviations away from the mean. We replace any standardised value beyond $\pm 5$ with 0, i.e. the training mean. Note that this is also a kind of imputation: we treat an outlier as a missing value.

In [ ]:
### replace outliers (more than 5 sd from the training mean) with 0, the training mean
def remove_outliers(F, sd_threshold=5):
    F = F.copy()
    outlier = np.abs(F) > sd_threshold
    F[outlier] = 0
    return F, outlier.sum()

F_train, n_out = remove_outliers(F_train)
F_test, _ = remove_outliers(F_test)
print('outliers replaced in the training set:', n_out)

We now train two models from scikit-learn. **LASSO logistic regression** is a linear model, $p(y=1|\mathbf{x}) = \sigma(\mathbf{w}^\top\mathbf{x}+b)$, with an $\ell_1$ penalty that pushes many weights to exactly zero, so it also selects features. Note that in scikit-learn, `C` is the *inverse* of the penalty strength.

A **balanced random forest** is a set of decision trees, where each tree is trained on a bootstrap sample that under-samples the majority class. This helps with the class imbalance (only 17% of the patients died); for logistic regression, `class_weight='balanced'` does the same job.

In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier

### train the two models on the summary features
clf_LR = LogisticRegression(penalty='l1', C=0.01, solver='saga', tol=1e-3, class_weight='balanced', random_state=SEED)
clf_LR.fit(F_train, y_train)
clf_RF = BalancedRandomForestClassifier(n_estimators=300, sampling_strategy='all', replacement=True, bootstrap=True,
                                        n_jobs=-1, random_state=SEED)
clf_RF.fit(F_train, y_train)

# test-set scores; we keep all the results in these two dicts, to compare the models in Part 4
results, probs = {}, {}
for name, clf in [('Logistic regression', clf_LR), ('Random forest', clf_RF)]:
    probs[name] = clf.predict_proba(F_test)[:, 1]
    results[name] = compute_scores(y_test, probs[name])
    print(name); print_scores(results[name])

In [ ]:
### which features matter? the LASSO coefficients and the random forest's feature importances
coef = clf_LR.coef_.ravel()
print(f'LASSO kept {np.sum(coef != 0)} of {len(coef)} features')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
top = np.argsort(np.abs(coef))[-15:]
ax1.barh(range(15), coef[top]); ax1.set_yticks(range(15)); ax1.set_yticklabels(F_names[top])
ax1.set_title('LASSO: 15 largest |coefficients|')
top = np.argsort(clf_RF.feature_importances_)[-15:]
ax2.barh(range(15), clf_RF.feature_importances_[top]); ax2.set_yticks(range(15)); ax2.set_yticklabels(F_names[top])
ax2.set_title('Random forest: 15 most important features')
fig.tight_layout(); plt.show()

**Does the imputer matter for the prediction?** The cell below repeats Part 2 with each of the three imputers. The hold-out MAE was quite different between the imputers; is the AUROC? Why, given how many values were missing? When would you expect the imputer to matter more?

In [ ]:
### repeat Part 2 with each imputer
rows = {}
for name, Xi in [('mean', X_mean), ('LOCF', X_locf), ('linear', X_lin)]:
    Fa = summary_features(Xi)
    Ftr, Fte = zscore(Fa[idx_train], Fa[idx_test])
    Ftr, Fte = remove_outliers(Ftr)[0], remove_outliers(Fte)[0]
    lr = LogisticRegression(penalty='l1', C=0.01, solver='saga', tol=1e-3, class_weight='balanced', random_state=SEED).fit(Ftr, y_train)
    rf = BalancedRandomForestClassifier(n_estimators=300, sampling_strategy='all', replacement=True, bootstrap=True,
                                        n_jobs=-1, random_state=SEED).fit(Ftr, y_train)
    rows[name] = {'LR AUROC': metrics.roc_auc_score(y_test, lr.predict_proba(Fte)[:, 1]),
                  'RF AUROC': metrics.roc_auc_score(y_test, rf.predict_proba(Fte)[:, 1])}
pd.DataFrame(rows).T.round(3)

## Part 3: sequence models (30 min)

Now we give the whole imputed sequence, $\mathbf{X}_n \in \mathbb{R}^{T\times P}$, to the three architectures from this morning's lecture. The task is *many-to-one*: one prediction for each patient.

| Model | Lecture section | How information is passed across time |
|---|---|---|
| LSTM | 2. Recurrent neural networks | a gated hidden state, updated step by step |
| TCN | 3. Temporal convolutional networks | dilated causal convolutions over a growing window |
| Transformer encoder | 4. Attention and Transformers | self-attention between all pairs of time steps |

### Pre-processing and the training loop (provided)
We pre-process the sequences as before, with all the statistics taken from the training patients: LOCF imputation (your TODO 3), clipping to ±5 SD, and then a z-score for each feature.

All three models use the same training loop, with `Adam` and `BCEWithLogitsLoss`. The models output a logit rather than a probability, and `pos_weight` gives more weight to the minority class. The loop also clips the gradients, which is the fix for exploding gradients that we saw in the lecture, and stops early when the validation AUROC stops improving, keeping the weights of the best epoch.

In [ ]:
### pre-process the sequences: LOCF imputation, clip to +-5 sd, then z-score each feature (statistics from the training set)
def preprocess_sequences(X, idx_train, train_mean, sd_threshold=5):
    X = impute_locf(X, train_mean)
    mu, sd = X[idx_train].mean(axis=(0, 1)), X[idx_train].std(axis=(0, 1)) + 1e-8
    X = np.clip(X, mu - sd_threshold * sd, mu + sd_threshold * sd)
    mu, sd = X[idx_train].mean(axis=(0, 1)), X[idx_train].std(axis=(0, 1)) + 1e-8
    return ((X - mu) / sd).astype(np.float32)

Xs = preprocess_sequences(X, idx_train, train_mean)
X_train, X_val, X_test = Xs[idx_train], Xs[idx_val], Xs[idx_test]
n_features = X.shape[2]
print(f'train {X_train.shape} | val {X_val.shape} | test {X_test.shape} | NaNs: {np.isnan(Xs).sum()}')

In [ ]:
# the number of trainable parameters of a model
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# the predicted probabilities of mortality, computed in batches
def predict_proba(model, X, batch_size=512):
    model.eval()
    with torch.no_grad():
        return np.concatenate([torch.sigmoid(model(torch.from_numpy(X[i:i + batch_size]).to(device))[:, 0]).cpu().numpy()
                               for i in range(0, len(X), batch_size)])


### the training loop: train on (X_train, y_train), with early stopping on the validation AUROC
def train(model, epochs=10, lr=1e-3, batch_size=64, patience=3):
    torch.manual_seed(SEED)
    model = model.to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    # up-weight the positive class (mortality) by the ratio of negative to positive patients
    pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype=torch.float32, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    Xt, yt = torch.from_numpy(X_train), torch.from_numpy(y_train.astype(np.float32))
    history, best_auc, best_state, wait, start = [], -1.0, None, 0, time.time()
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(Xt))   # shuffle the training patients at every epoch
        for i in range(0, len(Xt), batch_size):
            b = perm[i:i + batch_size]
            loss = loss_fn(model(Xt[b].to(device))[:, 0], yt[b].to(device))
            optimiser.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # gradient clipping
            optimiser.step()
        val_auc = metrics.roc_auc_score(y_val, predict_proba(model, X_val))
        history.append(val_auc)
        print(f'epoch {epoch:2d} | last batch loss {loss.item():.3f} | validation AUROC {val_auc:.3f} | {time.time() - start:.0f} s')
        # keep the best weights so far; stop if there is no improvement for `patience` epochs
        if val_auc > best_auc:
            best_auc, best_state, wait = val_auc, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                print(f'early stopping: no improvement for {patience} epochs')
                break
    model.load_state_dict(best_state)
    return model, {'val_auroc': history, 'seconds': time.time() - start}


# score a trained model on the test set, and save the results for Part 4
def record(name, model, history):
    probs[name] = predict_proba(model, X_test)
    results[name] = compute_scores(y_test, probs[name])
    results[name].update(params=count_parameters(model), seconds=history['seconds'])
    print(f'\n{name}, test set'); print_scores(results[name])

### LSTM
An LSTM keeps two vectors, a hidden state $\mathbf{h}_t$ and a cell state $\mathbf{c}_t$ (its memory). At each time step, it updates both from the new input $\mathbf{x}_t$ and the previous hidden state, using three gates with values between 0 and 1:

$$\begin{aligned}
\mathbf{f}_t &= \sigma(\mathbf{W}_f\mathbf{x}_t + \mathbf{U}_f\mathbf{h}_{t-1} + \mathbf{b}_f) && \text{forget gate: how much old memory to keep}\\
\mathbf{i}_t &= \sigma(\mathbf{W}_i\mathbf{x}_t + \mathbf{U}_i\mathbf{h}_{t-1} + \mathbf{b}_i) && \text{input gate: how much new information to store}\\
\tilde{\mathbf{c}}_t &= \tanh(\mathbf{W}_c\mathbf{x}_t + \mathbf{U}_c\mathbf{h}_{t-1} + \mathbf{b}_c) && \text{candidate memory}\\
\mathbf{c}_t &= \mathbf{f}_t\odot\mathbf{c}_{t-1} + \mathbf{i}_t\odot\tilde{\mathbf{c}}_t && \text{update the memory}\\
\mathbf{o}_t &= \sigma(\mathbf{W}_o\mathbf{x}_t + \mathbf{U}_o\mathbf{h}_{t-1} + \mathbf{b}_o) && \text{output gate: how much memory to show}\\
\mathbf{h}_t &= \mathbf{o}_t\odot\tanh(\mathbf{c}_t) && \text{new hidden state}
\end{aligned}$$

In PyTorch, `nn.LSTMCell` computes one step of these equations: `h, c = cell(x_t, (h, c))`. The next cell writes this step out by hand, gate by gate, and compares it with `nn.LSTMCell`. Run it, and try to match each line to one of the equations.

In [ ]:
### one LSTM step written out by hand (3 input features, hidden size 2)
cell = nn.LSTMCell(3, 2)
x_t, h, c = torch.randn(1, 3), torch.randn(1, 2), torch.randn(1, 2)
with torch.no_grad():
    z = x_t @ cell.weight_ih.T + h @ cell.weight_hh.T + cell.bias_ih + cell.bias_hh   # all four gates at once
    i, f, g, o = z.chunk(4, dim=-1)   # PyTorch stores them in this order: input, forget, candidate, output
    i, f, g, o = torch.sigmoid(i), torch.sigmoid(f), torch.tanh(g), torch.sigmoid(o)
    c_new = f * c + i * g   # update the memory
    h_new = o * torch.tanh(c_new)   # new hidden state
    h_ref, c_ref = cell(x_t, (h, c))
print('my step matches nn.LSTMCell:', torch.allclose(h_new, h_ref) and torch.allclose(c_new, c_ref))

### TODO 5: unroll the LSTM over time
A recurrent network applies the same cell, with the same weights, at every time step, and passes $(\mathbf{h}_t, \mathbf{c}_t)$ on to the next step. Writing this loop out over the 48 hours is what we called *unrolling* in the lecture.

Complete the loop in `forward()`. We start from $\mathbf{h}_0 = \mathbf{c}_0 = \mathbf{0}$. After the loop, `h` is the hidden state after the last step, $\mathbf{h}_T$, which summarises the whole series, and a linear layer maps it to one logit.

<b>Hint</b>: `x[:, t]` is time step `t` for all the patients in the batch, with shape `(batch, P)`.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.cell = nn.LSTMCell(n_features, hidden_dim)   # one time step: (x_t, (h, c)) -> (h, c)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):   # x: (batch, T, P)
        h = x.new_zeros(x.shape[0], self.hidden_dim)   # h_0 = 0, shape (batch, hidden)
        c = x.new_zeros(x.shape[0], self.hidden_dim)   # c_0 = 0
        ############################################################
        # TODO 5: write your code here.
        # Loop over the T time steps, and update (h, c) with self.cell at every step.
        # After the loop, h should be the hidden state after the last step, with shape (batch, hidden_dim).

        raise NotImplementedError('TODO 5: write your code above, then delete this line')
        ############################################################
        return self.fc(h)   # from the hidden state after the last step to one logit: (batch, 1)

In [ ]:
model = LSTMClassifier(n_features=3, hidden_dim=5)
x = torch.randn(2, 10, 3)   # 2 patients, 10 time steps, 3 features
out = model(x)
check('output shape (batch, 1)', out.shape == (2, 1))
x_first = x.clone(); x_first[:, 0] += 1.0
check('memory: changing the first time step changes the final output', not torch.allclose(model(x_first), out),
      'pass the (h, c) returned by one step into the next step')

# PyTorch's nn.LSTM runs the same loop (in faster code); copy our weights into it and compare
reference = nn.LSTM(3, 5, batch_first=True)
with torch.no_grad():
    for mine, theirs in [('weight_ih', 'weight_ih_l0'), ('weight_hh', 'weight_hh_l0'), ('bias_ih', 'bias_ih_l0'), ('bias_hh', 'bias_hh_l0')]:
        getattr(model.cell, mine).copy_(getattr(reference, theirs))
    _, (h_ref, _) = reference(x)
    check("with the same weights, matches PyTorch's nn.LSTM", torch.allclose(model(x), model.fc(h_ref[-1]), atol=1e-6),
          'visit every time step in order, t = 0, 1, ..., T - 1')

In [ ]:
### train the LSTM
lstm, hist_lstm = train(LSTMClassifier(n_features))
record('LSTM', lstm, hist_lstm)

### TCN
A TCN uses 1-D convolutions over time instead of a recurrence. `nn.Conv1d` expects the input as `(batch, channels, time)`, so the features become the channels.
- Causal convolution: we pad $(k-1)\cdot d$ zeros on the left only. The output at time $t$ then depends only on $\mathbf{x}_0,\dots,\mathbf{x}_t$, and it has the same length as the input.
- Dilation: the kernel skips $d-1$ steps between its taps. If we double $d$ at every level ($1, 2, 4, 8$), the window grows exponentially with the depth.

Each residual block has two dilated causal convolutions, each followed by normalisation, ReLU and dropout, and adds the block's input back to its output. As the network is causal, the output at the last time step has seen the whole history, so we classify from it.

### TODO 6: causal convolution
Complete `forward()`. <b>Hint</b>: `F.pad(x, (left, right))` pads the last dimension of `x`, which is time here.

In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.left_pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation)   # no padding here

    def forward(self, x):   # x: (batch, channels, T)
        ############################################################
        # TODO 6: write your code here.
        # Pad self.left_pad zeros on the left of the time dimension only, and then apply self.conv.
        # Return the output of the convolution, with shape (batch, out_channels, T).

        raise NotImplementedError('TODO 6: write your code above, then delete this line')
        ############################################################

In [ ]:
conv = CausalConv1d(3, 4, kernel_size=3, dilation=2)
x = torch.randn(1, 3, 20)
out = conv(x)
check(f'output length equals input length (yours: {out.shape[-1]})', out.shape == (1, 4, 20),
      'pad (kernel_size - 1) * dilation steps in total')
x_future = x.clone(); x_future[:, :, 10:] = torch.randn(1, 3, 10)
check('no leakage from the future: outputs before t = 10 ignore inputs from t = 10 on',
      out.shape == (1, 4, 20) and torch.allclose(conv(x_future)[:, :, :10], out[:, :, :10]),
      'all the padding must go on the left')

In [ ]:
### a residual block: two dilated causal convolutions, plus a skip connection
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(in_channels, out_channels, kernel_size, dilation), nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            CausalConv1d(out_channels, out_channels, kernel_size, dilation), nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout))
        # a 1x1 convolution on the skip path, if the number of channels changes
        self.skip = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return F.relu(self.net(x) + self.skip(x))


### the TCN: 4 blocks with dilation 1, 2, 4, 8, then a linear layer on the last time step
class TCNClassifier(nn.Module):
    def __init__(self, n_features, channels=64, levels=4, kernel_size=3, dropout=0.1):
        super().__init__()
        self.tcn = nn.Sequential(*[TemporalBlock(n_features if i == 0 else channels, channels, kernel_size, 2 ** i, dropout)
                                   for i in range(levels)])
        self.fc = nn.Linear(channels, 1)

    def forward(self, x):   # x: (batch, T, P)
        h = self.tcn(x.transpose(1, 2))   # (batch, channels, T)
        return self.fc(h[:, :, -1])   # the output at the last time step -> (batch, 1)

### TODO 7: receptive field
In the lecture, we saw that $L$ dilated causal layers with one convolution each, and $d = 1, 2, \dots, 2^{L-1}$, have a receptive field of $R = 1 + (k-1)(2^L - 1)$. Each of our residual blocks has *two* convolutions with the same dilation. Write a function for the receptive field of `TCNClassifier`. How many levels does it need, so that the last output sees all 48 hours?

<b>Hint</b>: the last output always sees its own time step, so start from $R = 1$. A causal convolution with kernel size $k$ and dilation $d$ lets each output see $(k-1)\cdot d$ more steps into the past. Then loop over the blocks: block $i$ (counting from 0) has dilation $d = 2^i$ and `convs_per_block` convolutions, so it adds `convs_per_block` $\times (k-1)\cdot 2^i$ steps to $R$. If you like, you can then turn the loop into a one-line formula like the lecture's, using $1 + 2 + \dots + 2^{L-1} = 2^L - 1$.

In [ ]:
def receptive_field(kernel_size, levels, convs_per_block=2):
    ############################################################
    # TODO 7: write your code here.
    # Start from R = 1, loop over the blocks, and add the number of steps that each block adds.
    # Return R (an integer): the number of time steps that the last output can see.

    raise NotImplementedError('TODO 7: write your code above, then delete this line')
    ############################################################

In [ ]:
for L in range(1, 6):
    print(f'levels = {L}: R = {receptive_field(3, L)}')
check('one conv per layer matches the lecture (k = 2, L = 4 -> 16)', receptive_field(2, 4, convs_per_block=1) == 16)
check('our TCN (k = 3, 4 levels, 2 convs per block) sees 61 steps', receptive_field(3, 4) == 61,
      'each block adds convs_per_block * (k - 1) * d steps')

In [ ]:
### train the TCN
tcn, hist_tcn = train(TCNClassifier(n_features))
record('TCN', tcn, hist_tcn)

### Transformer encoder
For classification, we only need the encoder half of the Transformer:
1. a linear embedding, which maps each time step $\mathbf{x}_t \in \mathbb{R}^P$ to a $d$-dimensional token;
2. a sinusoidal positional encoding $\mathbf{p}_t$, added to each token, as self-attention on its own ignores the order of the time steps;
3. Transformer blocks: multi-head self-attention, then a feed-forward network applied to each time step separately, both with a residual connection and (pre-)normalisation. We do not need a causal mask here, as we classify after seeing the whole series;
4. pooling: we average the 48 output tokens, and a linear layer gives one logit.

### TODO 8: scaled dot-product attention
The main part of the model is scaled dot-product attention:
$$\text{Attention}(\mathbf{Q},\mathbf{K},\mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_\text{head}}}\right)\mathbf{V}$$
The query of each time step is compared with the keys of all the time steps. The softmax turns these scores into weights that sum to 1, and the output is the weighted average of the values.

Write it as a function of `q`, `k` and `v`; you only need three lines, following the formula. You do not need to deal with the heads: the `Attention` module further down (provided) splits the tokens into heads, and calls your function on all of them at once.

<b>Hints</b>: `k.transpose(-2, -1)` swaps the last two dimensions (time steps and features), whatever dimensions come before them; `@` then multiplies the matrices in the last two dimensions. The softmax should normalise over the keys, i.e. the last dimension, so that the weights of each query sum to 1.

In [ ]:
# q, k, v: (..., T, d_head), where "..." can be any leading dimensions, e.g. (batch, heads)
# returns the output (..., T, d_head) and the attention weights (..., T, T)
def attention(q, k, v):
    d_head = q.shape[-1]
    ############################################################
    # TODO 8: write your code here, following the formula above.
    # scores: (..., T, T), one score for every (query, key) pair
    # weights: (..., T, T), the softmax of the scores, so that each row sums to 1
    # out: (..., T, d_head), the weighted average of the values
    scores = None
    weights = None
    out = None

    raise NotImplementedError('TODO 8: write your code above, then delete this line')
    ############################################################
    return out, weights

In [ ]:
q, k, v = torch.randn(5, 4), torch.randn(5, 4), torch.randn(5, 4)   # T = 5 time steps, d_head = 4
out, w = attention(q, k, v)
print('attention weights (rows: queries, columns: keys):\n', w.numpy().round(2))
check('shapes: output (T, d_head) and weights (T, T)', out.shape == (5, 4) and w.shape == (5, 5))
check('each row of the weights sums to 1', w.shape == (5, 5) and torch.allclose(w.sum(-1), torch.ones(5)),
      'softmax over the keys: dim=-1')

# now with (batch, heads, T, d_head), compared with PyTorch's own function
q, k, v = torch.randn(3, 2, 10, 4), torch.randn(3, 2, 10, 4), torch.randn(3, 2, 10, 4)
try:
    matches = torch.allclose(attention(q, k, v)[0], F.scaled_dot_product_attention(q, k, v), atol=1e-5)
except RuntimeError:   # e.g. a shape error
    matches = False
check("works on (batch, heads, T, d_head) and matches PyTorch's F.scaled_dot_product_attention", matches,
      'use k.transpose(-2, -1) rather than k.T, and divide the scores by sqrt(d_head) before the softmax')

In [ ]:
### multi-head self-attention: project to Q, K, V, split into heads, call your attention() on all heads, then merge the heads
class Attention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=16, dropout=0.1):
        super().__init__()
        self.heads = heads
        self.to_qkv = nn.Linear(dim, 3 * heads * dim_head, bias=False)   # W_Q, W_K, W_V for all heads
        self.to_out = nn.Sequential(nn.Linear(heads * dim_head, dim), nn.Dropout(dropout))   # W_O

    def forward(self, x):   # x: (batch, T, dim)
        b, T, _ = x.shape
        q, k, v = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = [t.reshape(b, T, self.heads, -1).transpose(1, 2) for t in (q, k, v)]   # (batch, heads, T, d_head)
        out, weights = attention(q, k, v)   # your TODO 8
        self.last_attn = weights.detach()   # we keep these for the plot below
        out = out.transpose(1, 2).reshape(b, T, -1)   # concatenate the heads
        return self.to_out(out)

In [ ]:
### the fixed sinusoidal positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, dim, max_len=500):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2) * (-math.log(10000.0) / dim))
        pe = torch.zeros(max_len, dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)   # fixed, not trained

    def forward(self, x):   # x: (batch, T, dim)
        return x + self.pe[:x.size(1)]


### one Transformer block (pre-norm): self-attention, then a feed-forward network
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, dim_head, ff_dim, dropout):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads, dim_head, dropout)
        self.norm2, self.ff = nn.LayerNorm(dim), nn.Sequential(
            nn.Linear(dim, ff_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff_dim, dim), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))   # mixes information across the time steps
        return x + self.ff(self.norm2(x))   # works on each time step on its own


### the Transformer encoder for classification
class TransformerClassifier(nn.Module):
    def __init__(self, n_features, dim=64, depth=2, heads=4, dim_head=16, ff_dim=128, dropout=0.1, use_pe=True):
        super().__init__()
        self.embedding = nn.Linear(n_features, dim)
        self.pos_encoding = PositionalEncoding(dim) if use_pe else nn.Identity()
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[TransformerBlock(dim, heads, dim_head, ff_dim, dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.fc = nn.Linear(dim, 1)

    def forward(self, x):   # x: (batch, T, P)
        h = self.dropout(self.pos_encoding(self.embedding(x)))   # (batch, T, dim)
        h = self.norm(self.blocks(h))   # (batch, T, dim)
        return self.fc(h.mean(dim=1))   # average over time, then one logit -> (batch, 1)

# train the Transformer
tfm, hist_tfm = train(TransformerClassifier(n_features))
record('Transformer', tfm, hist_tfm)

In [ ]:
### where does the model look? the attention weights of the last block, for the test patient with the highest predicted risk
n = int(np.argmax(probs['Transformer']))
tfm.eval()
with torch.no_grad():
    tfm(torch.from_numpy(X_test[n:n + 1]).to(device))
attn = tfm.blocks[-1].attn.last_attn[0].cpu().numpy()   # (heads, T, T)
fig, axes = plt.subplots(1, attn.shape[0], figsize=(4 * attn.shape[0], 3.6))
for h, ax in enumerate(axes):
    ax.imshow(attn[h], cmap='Blues')
    ax.set_title(f'head {h + 1}'); ax.set_xlabel('attends to (keys)'); ax.set_ylabel('time step (queries)')
fig.suptitle(f"Test patient {n} (predicted risk {probs['Transformer'][n]:.2f}, outcome: {'died' if y_test[n] else 'survived'})")
fig.tight_layout(); plt.show()

## Part 4: compare all the models (10 min)

The test set has about 1,400 patients, so each AUROC is a bit noisy. To see how much, the table below gives a bootstrap 95% confidence interval: we resample the test patients with replacement many times, compute the AUROC each time, and take the 2.5th and 97.5th percentiles.

In [ ]:
### bootstrap 95% confidence interval of the AUROC
def bootstrap_ci(y_true, y_prob, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y_true), len(y_true))
        if y_true[i].min() != y_true[i].max():   # skip a resample with only one class
            aucs.append(metrics.roc_auc_score(y_true[i], y_prob[i]))
    return np.percentile(aucs, [2.5, 97.5])

# the comparison table
rows = {}
for name, s in results.items():
    lo, hi = bootstrap_ci(y_test, probs[name])
    rows[name] = {'AUROC': f"{s['AUROC']:.3f}", '95% CI': f'{lo:.3f} to {hi:.3f}',
                  'AUPRC': f"{s['AUPRC']:.3f}",
                  'parameters': f"{s['params']:,}" if 'params' in s else '',
                  'train time (s)': f"{s['seconds']:.0f}" if 'seconds' in s else ''}
pd.DataFrame(rows).T

In [ ]:
### the validation AUROC of the three sequence models during training
plt.figure(figsize=(6, 4))
for name, h in [('LSTM', hist_lstm), ('TCN', hist_tcn), ('Transformer', hist_tfm)]:
    plt.plot(h['val_auroc'], 'o-', label=name)
plt.xlabel('Epoch'); plt.ylabel('Validation AUROC'); plt.grid(alpha=0.3); plt.legend(); plt.show()

**Discuss:**
1. Do the sequence models beat the random forest on summary features? Do the confidence intervals overlap? Is the extra complexity worth it here?
2. In the lecture, we said that TCNs and Transformers can train faster than RNNs, as they process all the time steps in parallel. Is this what the `train time` column shows? Why might we not see it here (48 steps, small models, maybe a CPU), and when would we?
3. All the models were given the imputed values as if they had been measured. How could we tell a model which values were imputed? Would that help on this dataset, where the values are missing completely at random? And on real EHR data?

## Extensions (optional)

**E1. Many more missing values.** Use `holdout_mask(X, frac=0.3)` to hide many more values, and repeat the comparison of the imputers. When does the choice of imputer start to change the AUROC?

**E2. Other architectures from the lecture.**
- GRU: write one GRU step by hand from the equations in the lecture, with a reset gate $\mathbf{r}_t$, an update gate $\mathbf{z}_t$ and a candidate $\tilde{\mathbf{h}}_t$, and no separate cell state. Unroll it over time as in TODO 5, and compare its AUROC, number of parameters and training time with your LSTM. Then try `nn.GRU(n_features, 64, batch_first=True)`, which runs the same loop in faster code. Note that PyTorch's version swaps the roles of $\mathbf{z}_t$ and $1 - \mathbf{z}_t$, and applies the reset gate after the matrix product, so its numbers will not match your step exactly.
- MS-TCN and DC-TCN: in a multi-scale TCN (MS-TCN), each convolution in `TemporalBlock` is replaced by parallel branches with kernel sizes $k = 3, 5, 7$ (one `CausalConv1d` each, with the channels split between them), and their outputs are concatenated. In a densely connected TCN (DC-TCN), each layer in a block receives the block input together with the outputs of all the earlier layers, and each layer has its own kernel size and dilation; you can also add squeeze-and-excitation (SE) attention over the channels. Build one or both, and compare their receptive fields (adapt `receptive_field()` from TODO 7), AUROC and number of parameters with the plain TCN. If you want to have a look, the official implementations are at [github.com/mpc001/Lipreading_using_Temporal_Convolutional_Networks](https://github.com/mpc001/Lipreading_using_Temporal_Convolutional_Networks), in `lipreading/models/tcn.py` (MS-TCN) and `lipreading/models/densetcn.py` (DC-TCN). They were written for lip reading, where the model sees the whole video clip at once, so their convolutions pad on both sides and are not causal; use our `CausalConv1d` instead if you borrow from them.
- Encoder-decoder: use the full Transformer from the lecture to *forecast* instead of classify. The encoder reads the first 36 hours, and the decoder predicts the last 12 hours of a few vital signs, e.g. heart rate, mean blood pressure and respiratory rate. The decoder needs two new parts: masked self-attention (add a `mask` argument to your `attention()`, which sets the scores of future time steps to $-\infty$ before the softmax), and cross-attention, where the queries come from the decoder and the keys and values from the encoder's output. During training, give the decoder the true values shifted by one step, after a start token (teacher forcing). Compare its MAE with simply repeating the last observed value; `nn.Transformer` can be used as a reference.